# t5 - us dataset
fine-tuning t5-base on stanford congressional speech
t5 is encoder-decoder so the label is generated as text, not predicted from a softmax head

In [ ]:
# t5 needs sentencepiece
! pip install transformers scikit-learn sentencepiece

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
# t5 needs its own tokenizer and conditional generation class
from transformers import T5Tokenizer, T5ForConditionalGeneration, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score
print('imports done')

In [ ]:
# gpu
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# file paths
trainPath ='/kaggle/input/datasets/zainabrizvi5/thesis-data/final_dataset/us_train.csv'
valPath = '/kaggle/input/datasets/zainabrizvi5/thesis-data/final_dataset/us_val.csv'
testPath= '/kaggle/input/datasets/zainabrizvi5/thesis-data/final_dataset/us_test.csv'
output_dir= '/kaggle/working/t5_us'

In [ ]:
model_name = 't5-base'
max_length= 256
batch_size= 16
epochs= 3
lr = 2e-5
# if t5 generates something unexpected, we default to the first label
valid_labels =['left', 'right']

In [ ]:
#labels - also used to validate generated output
label2id ={'left': 0, 'right': 1}
id2label= {0: 'left', 1: 'right'}

In [ ]:
#load data
train_df= pd.read_csv(trainPath)
val_df= pd.read_csv(valPath)
test_df  = pd.read_csv(testPath)
print(f'train: {len(train_df)}, val: {len(val_df)}, test: {len(test_df)}')

In [ ]:
#t5 dataset formats input as a prompt and target as the label word
class SpeechDataset(Dataset):
    # t5 is text-to-text so we format input as a prompt and target as the label word
    def __init__(self, df, tokenizer, max_length, label2id):
        self.texts = df['text'].tolist()
        self.labels= df['label'].tolist()
        self.tokenizer= tokenizer
        self.max_length= max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        input_text= f'classify political ideology: {self.texts[idx]}'
        target_text= self.labels[idx]
        enc= self.tokenizer(input_text, max_length=self.max_length,padding='max_length', truncation=True, return_tensors='pt')
        tgt = self.tokenizer(target_text, max_length=10, padding='max_length', truncation=True, return_tensors='pt')
        # mask padding tokens in labels
        labs = tgt['input_ids'].squeeze()
        labs[labs==self.tokenizer.pad_token_id] = -100
        return {'input_ids':enc['input_ids'].squeeze(),'attention_mask':enc['attention_mask'].squeeze(),'labels':labs}

In [ ]:
# t5 tokenizer
tokenizer =T5Tokenizer.from_pretrained(model_name)

In [ ]:
# create datasets
train_dataset= SpeechDataset(train_df,tokenizer, max_length,label2id)
val_dataset= SpeechDataset(val_df,tokenizer, max_length,label2id)
test_dataset= SpeechDataset(test_df,tokenizer, max_length, label2id)

In [ ]:
# dataloaders
train_loader= DataLoader(train_dataset,batch_size=batch_size, shuffle=True)
val_loader=DataLoader(val_dataset,batch_size=batch_size)
test_loader= DataLoader(test_dataset, batch_size=batch_size)

In [ ]:
# t5 for conditional generation — no classification head
model = T5ForConditionalGeneration.from_pretrained(model_name)
model.to(device)
print('model loaded')

In [ ]:
# optimizer and scheduler
optimizer= AdamW(model.parameters(), lr=lr)
total_steps= len(train_loader) * epochs
scheduler= get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

In [ ]:
# evaluation decodes generated tokens and compares to true label
def evaluate(model, loader, valid_labels, split='val'):
    model.eval()
    all_preds, all_labels=[], []
    with torch.no_grad():
        for batch in loader:
            ids = batch['input_ids'].to(device)
            mask= batch['attention_mask'].to(device)
            labs = batch['labels'].to(device)
            gen= model.generate(input_ids=ids, attention_mask=mask, max_new_tokens=5)
            for g,l in zip(gen, labs):
                pred =tokenizer.decode(g, skip_special_tokens=True).strip().lower()
                # fallback if model generates something unexpected
                if pred not in valid_labels:
                    pred = valid_labels[0]
                true_ids= l[l != -100]
                true = tokenizer.decode(true_ids, skip_special_tokens=True).strip().lower()
                all_preds.append(pred)
                all_labels.append(true)
    acc= accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')
    print(f'{split} — acc: {acc:.4f}, macro f1: {f1:.4f}')
    return acc, f1

In [ ]:
# training loop
history = []
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        ids= batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        labs= batch['labels'].to(device)
        out= model(input_ids=ids, attention_mask=mask, labels=labs)
        loss =out.loss
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss+= loss.item()
    avg_loss= total_loss / len(train_loader)
    val_acc,val_f1 =evaluate(model, val_loader, valid_labels, split=f'epoch {epoch+1} val')
    history.append({'epoch': epoch+1, 'train_loss': avg_loss, 'val_acc': val_acc, 'val_f1': val_f1})
    print(f'epoch {epoch+1} train loss: {avg_loss:.4f}')

In [ ]:
# print history
print('training done')
for h in history:
    print(h)

In [ ]:
# test set evaluation
print('final test evaluation')
test_acc, test_f1 = evaluate(model,test_loader,valid_labels,split='test')

In [ ]:
# save model and results
pd.DataFrame(history).to_csv(os.path.join(output_dir, 'training_history.csv'),index=False)
pd.DataFrame([{'model': 't5-base', 'dataset': 'US (Stanford)','test_acc': test_acc,'test_f1': test_f1,'epochs':epochs, 'batch_size': batch_size, 'lr': lr, 'max_length':max_length}]).to_csv(os.path.join(output_dir,'final_results.csv'),index=False)
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f'saved to {output_dir}')